### If you get an error here, it means you need to run the prepare_data.ipynb file first

In [1]:
import json
import numpy as np
import pandas as pd

from rnanorm import TMM
from pathlib import Path


description_path = Path('description.json')
if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    data = json.load(f)

## Preprocessing steps

1. **Load data** — Read the dataset and split columns into clinical variables and raw gene-level counts.

2. **Robust-expression filter (defined on raw counts)** — Keep genes with the 75th percentile of raw counts across samples \(\ge 100\) (i.e., expressed at \(\ge 100\) counts in at least 25% of samples).

3. **TMM normalization** — Apply TMM to the full gene count matrix to compute normalization factors correcting for library size and RNA composition; obtain normalized counts.

4. **Apply gene filter and log-transform** — Subset to the robustly expressed genes and apply \(\log(x+1)\).

5. <strike>**Z-score standardization** — For each gene, center and scale across samples to mean 0 and standard deviation 1.</strike>

In [2]:
data_file_name = data['data_file_names'][0]

num_clinical_features = data['data_summary']['clinical_features']

df = pd.read_csv(data_file_name)


clinical_cols = df.columns[:num_clinical_features]
gene_cols = df.columns[num_clinical_features:]

raw_counts = df[gene_cols]
print('Before preprocessing:\n - n_genes: %d' % (df[gene_cols].shape[1]))

# create mask of robustly expressed genes
robus_genes_mask = np.percentile(df[gene_cols], 75, axis=0) >= 100

# normalize counts
df[gene_cols] = TMM().set_output(transform="pandas").fit_transform(df[gene_cols])

# apply robust gene mask
gene_cols = df[gene_cols].columns[robus_genes_mask].tolist()
df = df[list(clinical_cols) + gene_cols]

# # log transform and standardize (genes only)
df[gene_cols] = np.log(df[gene_cols] + 1)
# df[gene_cols] = (df[gene_cols] - df[gene_cols].mean(axis=0)) / df[gene_cols].std(axis=0)

print("\nAfter preprocessing:\n - n_genes: %d" % (df[gene_cols].shape[1]))

transformed_data_file_name = data_file_name.replace("_raw_counts_NCBI", "_preprocessed")


/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_23337/2835454030.py:5: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_file_name)


Before preprocessing:
 - n_genes: 39376



After preprocessing:
 - n_genes: 15641


In [3]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
df_gene_annotations = df_gene_annotations[df_gene_annotations["Status"] == "active"]
df_gene_annotations = df_gene_annotations[df_gene_annotations["GeneType"] == "protein-coding"]
df_gene_annotations = df_gene_annotations.reset_index(drop=True)

protein_coding_genes = set(df_gene_annotations["Symbol"].dropna())
gene_cols = [gene for gene in gene_cols if gene in protein_coding_genes]
df = df[list(clinical_cols) + gene_cols]

print("\nAfter protein-coding filter:\n - n_genes: %d" % len(gene_cols))

df.to_csv(transformed_data_file_name, index=False)

if transformed_data_file_name not in data['data_file_names']:
    data['data_file_names'].append(transformed_data_file_name)

with open(description_path, 'w') as f:
    json.dump(data, f, indent=4)

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_23337/4029965179.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(



After protein-coding filter:
 - n_genes: 13389


## Survival variables encodings

Authors encode survival variables in various ways, here we have just a check if all the survival variables are already numeric. If they are not, we convert them by hand.


In [4]:
df_ = df.copy()

endpoints = []
for endpoint in data["survival-endpoints"]:
    endpoint_time = endpoint["time_var"]["var_name"]
    endpoint_event = endpoint["event_var"]["var_name"]

    if endpoint_time == "unknown" or endpoint_event == "unknown":
        continue

    # Replace "Not available" and similar missing-value strings with NaN
    for col in [endpoint_time, endpoint_event]:
        df_[col] = df_[col].replace(["Not available", "NA", "N/A", "not available", "-"], pd.NA)


    # make sure time and event are numeric
    df_[endpoint_time] = pd.to_numeric(df_[endpoint_time])
    df_[endpoint_event] = pd.to_numeric(df_[endpoint_event])
    
    endpoints.append((endpoint_time, endpoint_event))

# Flatten endpoints and use it to order survival columns
surv_cols = [col for tup in endpoints for col in tup]
other_cols = [col for col in df_.columns if col not in surv_cols]
ordered_cols = surv_cols + other_cols

df_[ordered_cols].to_csv(transformed_data_file_name, index=False)

In [5]:
df_[ordered_cols]

,os time (months),OS status,sample_title,tissue,age(0,genotype(1,tumor size (0,t stage (1,n stage (0,m stage (0,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,93.700000,0.0,TFE3-1,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,...,5.308855,3.757621,5.360897,5.656186,3.962839,4.470927,6.469223,6.748252,6.125305,5.764987
1,104.800000,0.0,TFE3-2,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 3,n0; 1:n1): 0,m0; 1:m1): 0,...,5.848443,3.730599,5.919254,6.527469,4.660299,4.747210,6.781149,6.862246,4.832972,6.616324
2,52.333333,1.0,TFE3-3,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 1,...,7.678366,5.906810,7.449775,7.975221,5.812916,6.476641,8.494724,7.960829,6.346937,7.731272
3,71.900000,0.0,TFE3-4,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 1,m0; 1:m1): 0,...,6.451027,4.863556,6.369385,6.755879,5.045029,6.036748,7.921962,8.600118,7.713849,7.892062
4,10.000000,1.0,TFE3-5,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 4,n0; 1:n1): 0,m0; 1:m1): 0,...,6.448179,4.869214,6.499521,7.077482,4.883677,5.596320,7.486865,7.328116,6.006584,6.856885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,29.000000,0.0,TFE3-65,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,...,7.532142,5.796261,7.331172,7.777613,5.695442,6.152159,8.005513,7.544120,5.958480,7.317851
57,3.000000,0.0,TFE3-66,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,...,7.273212,5.607325,7.084405,7.431970,5.154626,6.132811,8.117052,8.113245,6.918626,7.320884
58,2.000000,0.0,TFE3-67,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 1,m0; 1:m1): 0,...,8.072118,6.890950,8.349514,8.564182,6.787443,7.099458,9.128801,8.782960,7.294296,8.664169
59,28.800000,0.0,TFE3-69,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 2,n0; 1:n1): 0,m0; 1:m1): 0,...,6.107800,4.426978,6.056608,7.067310,4.240064,5.143558,7.067854,7.031471,5.351965,6.630987
